# Force vs extension

On the Tk console, write this chunk of code:

In [ ]:
# Function to read GROMACS xvg files
proc read_xvg {filename col} {
    set data {}
    set fd [open $filename r]
    while {[gets $fd line] >= 0} {
        if {[string index $line 0] != "@" && [string index $line 0] != "#"} {
            lappend data [lindex $line $col]
        }
    }
    close $fd
    return $data
}

# 1. Load the Force (from column 1, which is the 2nd column)
set force [read_xvg "smd_replica1_pullf.xvg" 1]

# 2. Load the Position and convert to Extension
set pos [read_xvg "smd_replica1_pullx.xvg" 1]
set start_pos [lindex $pos 0]
set extension {}
foreach p $pos {
    # Calculate extension relative to start and convert nm to Angstroms (multiply by 10)
    lappend extension [expr ($p - $start_pos) * 10.0]
}

The previous script will read the `xvg` files and store the values in lists that VMD can understand. Now, the data is loaded into the variables $extension and $force, run the following script:

In [ ]:
package require multiplot
set plot1 [multiplot -x $extension -y $force -title "Force vs Extension" -xlabel "Extension (A)" -ylabel "Force (kJ/mol/nm)" -linecolor red -plot]

# Work vs extension

To do the Work vs Extension plot, we need to integrate the force by running the following code:

In [ ]:
set work {0.0}
set cumulative_w 0.0
for {set i 1} {$i < [llength $force]} {incr i} {
    set dx [expr [lindex $extension $i] - [lindex $extension [expr $i-1]]]
    set avg_f [expr ([lindex $force $i] + [lindex $force [expr $i-1]]) / 2.0]
    # Work in kJ/mol (note: dx is in A, so we divide by 10 to get nm)
    set cumulative_w [expr $cumulative_w + ($avg_f * ($dx / 10.0))]
    lappend work $cumulative_w
}

# Plot the Work
set plot2 [multiplot -x $extension -y $work -title "Work vs Extension (PMF)" -xlabel "Extension (A)" -ylabel "Work (kJ/mol)" -linecolor blue -plot]

- Force vs. Extension: Shows the "strength" of the binding pocket.

- Work vs. Extension: Shows the "energy cost" to unbind the ligand (PMF).

- Time vs. Extension: Shows the "kinetics" or how smoothly/abruptly the ligand escaped.

# Time vs Extension

In [ ]:
# 1. Load the Time (from column 0 of the pullx file)
set time [read_xvg "smd_replica1_pullx.xvg" 0]

# 2. Plot Time vs Extension
package require multiplot
set plot3 [multiplot -x $time -y $extension -title "Time vs Extension" -xlabel "Time (ps)" -ylabel "Extension (A)" -linecolor green -plot]